# DATA CLEANING

## Loading the IBM AML transaction dataset

In [ ]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv(r"C:\Users\tjain\Downloads\HI-Small_Trans.csv")

print("Dataset loaded!")
print(f"Shape: {df.shape}")

In [5]:
df.head(10)

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0
5,2022/09/01 00:03,1,8000F5AD0,1,8000F5AD0,6162.44,US Dollar,6162.44,US Dollar,Reinvestment,0
6,2022/09/01 00:08,1,8000EBAC0,1,8000EBAC0,14.26,US Dollar,14.26,US Dollar,Reinvestment,0
7,2022/09/01 00:16,1,8000EC1E0,1,8000EC1E0,11.86,US Dollar,11.86,US Dollar,Reinvestment,0
8,2022/09/01 00:26,12,8000EC280,2439,8017BF800,7.66,US Dollar,7.66,US Dollar,Credit Card,0
9,2022/09/01 00:21,1,8000EDEC0,211050,80AEF5310,383.71,US Dollar,383.71,US Dollar,Credit Card,0


In [ ]:
## Calculating total rows and columns

In [13]:
df.shape

(5078336, 15)

## looking at what we are working 

In [6]:
print("Column names:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())

Column names: ['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']

Data types:
Timestamp              object
From Bank               int64
Account                object
To Bank                 int64
Account.1              object
Amount Received       float64
Receiving Currency     object
Amount Paid           float64
Payment Currency       object
Payment Format         object
Is Laundering           int64
dtype: object

Missing values:
Timestamp             0
From Bank             0
Account               0
To Bank               0
Account.1             0
Amount Received       0
Receiving Currency    0
Amount Paid           0
Payment Currency      0
Payment Format        0
Is Laundering         0
dtype: int64


## Fix column names

In [9]:
# Renaming all columns to clear meaningful names
# Account   → Sender Account
# Account.1 → Receiver Account
# From Bank → Sender Bank ID
# To Bank   → Receiver Bank ID

df = df.rename(columns={
    'Account'   : 'Sender Account',    # who sent money
    'Account.1' : 'Receiver Account',  # who received money
    'From Bank' : 'Sender Bank ID',    # bank that sent
    'To Bank'   : 'Receiver Bank ID'   # bank that received
})

print("Renamed columns:")
for col in df.columns:
    print(f"  {col}")

Renamed columns:
  Timestamp
  Sender Bank ID
  Sender Account
  Receiver Bank ID
  Receiver Account
  Amount Received
  Receiving Currency
  Amount Paid
  Payment Currency
  Payment Format
  Is Laundering


## Fix the Timestamp column

In [10]:
# Right now Timestamp is stored as TEXT (object)
# Example: "2022/09/01 00:20"
# We need to convert it to a real datetime so we can
# extract hour, day, day-of-week etc. later

df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%Y/%m/%d %H:%M')

# Extract useful time features RIGHT NOW while we are here
df['Hour']       = df['Timestamp'].dt.hour        # 0-23
df['Day']        = df['Timestamp'].dt.day          # 1-31
df['DayOfWeek']  = df['Timestamp'].dt.dayofweek   # 0=Monday, 6=Sunday
df['IsWeekend']  = (df['DayOfWeek'] >= 5).astype(int)  # 1=weekend, 0=weekday

# Check it worked
print("Timestamp sample:")
print(df[['Timestamp','Hour','Day','DayOfWeek','IsWeekend']].head(3))

Timestamp sample:
            Timestamp  Hour  Day  DayOfWeek  IsWeekend
0 2022-09-01 00:20:00     0    1          3          0
1 2022-09-01 00:20:00     0    1          3          0
2 2022-09-01 00:00:00     0    1          3          0


## checking for missing values

In [11]:
# Count missing values per column
missing = df.isnull().sum()

print("Missing values per column:")
print(missing)
print(f"\nTotal missing values: {missing.sum()}")

Missing values per column:
Timestamp             0
Sender Bank ID        0
Sender Account        0
Receiver Bank ID      0
Receiver Account      0
Amount Received       0
Receiving Currency    0
Amount Paid           0
Payment Currency      0
Payment Format        0
Is Laundering         0
Hour                  0
Day                   0
DayOfWeek             0
IsWeekend             0
dtype: int64

Total missing values: 0


#### Therefore no missing values

## checking for duplicate rows

In [12]:
# Duplicate rows can confuse the ML model
# Same transaction appearing twice = wrong

duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates}")

# If there are duplicates, remove them
if duplicates > 0:
    df = df.drop_duplicates()
    print(f"Duplicates removed. New shape: {df.shape}")
else:
    print("No duplicates found — good!")

Duplicate rows: 9
Duplicates removed. New shape: (5078336, 15)


## Check the cross bank tranfer and self account transfer flags ⭐⭐

In [16]:
# These are important fraud signals we discussed
# is_cross_bank   → sender and receiver are at different banks
# is_self_transfer → sender and receiver are the same account

df['is_cross_bank']    = (df['Sender Bank ID'] != df['Receiver Bank ID']).astype(int)
df['is_self_transfer'] = (df['Sender Account'] == df['Receiver Account']).astype(int)

# Check the counts
print("Cross bank transfers:")
print(df['is_cross_bank'].value_counts())

print("\nSelf transfers:")
print(df['is_self_transfer'].value_counts())

Cross bank transfers:
is_cross_bank
1    4387008
0     691328
Name: count, dtype: int64

Self transfers:
is_self_transfer
0    4487124
1     591212
Name: count, dtype: int64


## Check unique values in text columns

In [17]:
# We need to know all unique values in text columns
# before we encode them to numbers

print("Payment Format values:")
print(df['Payment Format'].value_counts())

print("\nPayment Currency values:")
print(df['Payment Currency'].value_counts())

print("\nReceiving Currency values:")
print(df['Receiving Currency'].value_counts())

Payment Format values:
Payment Format
Cheque          1864331
Credit Card     1323324
ACH              600793
Cash             490891
Reinvestment     481056
Wire             171855
Bitcoin          146086
Name: count, dtype: int64

Payment Currency values:
Payment Currency
US Dollar            1895169
Euro                 1168296
Swiss Franc           234860
Yuan                  213752
Shekel                192184
Rupee                 190202
UK Pound              180738
Yen                   155209
Ruble                 155178
Bitcoin               146061
Canadian Dollar       140042
Australian Dollar     136769
Mexican Peso          110159
Saudi Riyal            89014
Brazil Real            70703
Name: count, dtype: int64

Receiving Currency values:
Receiving Currency
US Dollar            1879341
Euro                 1172017
Swiss Franc           237884
Yuan                  206551
Shekel                194988
Rupee                 192065
UK Pound              181255
Ruble         

## Encode text columns to numbers

In [18]:
# ML models only understand numbers, not text
# We convert each text category to a number
# This is called Label Encoding

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

# Encode Payment Format (Wire, Cheque, Cash etc.)
df['Payment_Format_Encoded'] = le.fit_transform(df['Payment Format'])
print("Payment Format encoding:")
for original, encoded in zip(le.classes_, range(len(le.classes_))):
    print(f"  {original} → {encoded}")

# Encode Payment Currency
df['Payment_Currency_Encoded'] = le.fit_transform(df['Payment Currency'])

# Encode Receiving Currency
df['Receiving_Currency_Encoded'] = le.fit_transform(df['Receiving Currency'])

print("\nEncoding done!")

Payment Format encoding:
  ACH → 0
  Bitcoin → 1
  Cash → 2
  Cheque → 3
  Credit Card → 4
  Reinvestment → 5
  Wire → 6

Encoding done!


## Create currency mismatch ⭐⭐

In [19]:
# If Payment Currency != Receiving Currency
# money was converted mid-transfer
# This is a classic laundering technique

df['is_currency_mismatch'] = (
    df['Payment Currency'] != df['Receiving Currency']
).astype(int)

print("Currency mismatch count:")
print(df['is_currency_mismatch'].value_counts())

print("\nMismatch in fraud vs normal:")
print(df.groupby('Is Laundering')['is_currency_mismatch'].mean().round(4))

Currency mismatch count:
is_currency_mismatch
0    5006170
1      72166
Name: count, dtype: int64

Mismatch in fraud vs normal:
Is Laundering
0    0.0142
1    0.0000
Name: is_currency_mismatch, dtype: float64


But label encoding confuse the model with priority like 0>1>2>3>4>5.... but these are categories not priorities

Therefore one hot hot encoding works here

But the problem with one hot encoding is that :

for every unique value i need to create a new column and in this project there are:

The smart decision for this project
                  
Payment Format → One Hot Encoding because it has only 6 unique values (Cheque, Cash, Wire, ACH, Reinvestment, Bitcoin) and no order between them. This adds only 6 columns and is completely safe.
Payment Currency and Receiving Currency → Label Encoding is actually fine because XGBoost internally uses decision trees that split on values like is currency == 3 — it does NOT actually treat higher numbers as more important the way linear models do. The false ranking problem mainly affects linear regression and neural networks, not tree-based models like XGBoost.
Sender Account and Receiver Account → Never OHE because they have 500,000+ unique values. OHE would create 500,000 new columns and crash your entire notebook instantly.

## Replace label encoding with this

In [22]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

# ── Payment Format → ONE HOT ENCODING ──────────────────────────
# Only 6 unique values — safe to OHE
# pd.get_dummies creates one column per category automatically

payment_format_ohe = pd.get_dummies(
    df['Payment Format'],
    prefix='fmt'        # column names become fmt_Cash, fmt_Wire etc.
)

# Join the new columns to our dataframe
df = pd.concat([df, payment_format_ohe], axis=1)

print("Payment Format OHE columns created:")
print(payment_format_ohe.columns.tolist())

# ── Payment Currency → LABEL ENCODING ──────────────────────────
# 20+ currencies — OHE would add too many columns
# XGBoost handles this fine with label encoding

df['Payment_Currency_Encoded'] = le.fit_transform(df['Payment Currency'])
print("\nPayment Currency encoded — unique values:")
print(le.classes_)

# ── Receiving Currency → LABEL ENCODING ────────────────────────
df['Receiving_Currency_Encoded'] = le.fit_transform(df['Receiving Currency'])
print("\nReceiving Currency encoded!")

# ── Verify new columns ──────────────────────────────────────────
fmt_cols = [c for c in df.columns if c.startswith('fmt_')]
print(f"\nNew OHE columns ({len(fmt_cols)}):", fmt_cols)
print(f"Total columns now: {df.shape[1]}")

Payment Format OHE columns created:
['fmt_ACH', 'fmt_Bitcoin', 'fmt_Cash', 'fmt_Cheque', 'fmt_Credit Card', 'fmt_Reinvestment', 'fmt_Wire']

Payment Currency encoded — unique values:
['Australian Dollar' 'Bitcoin' 'Brazil Real' 'Canadian Dollar' 'Euro'
 'Mexican Peso' 'Ruble' 'Rupee' 'Saudi Riyal' 'Shekel' 'Swiss Franc'
 'UK Pound' 'US Dollar' 'Yen' 'Yuan']

Receiving Currency encoded!

New OHE columns (7): ['fmt_ACH', 'fmt_Bitcoin', 'fmt_Cash', 'fmt_Cheque', 'fmt_Credit Card', 'fmt_Reinvestment', 'fmt_Wire']
Total columns now: 28


## Create flag mismatch ⭐⭐

In [23]:
# If Payment Currency != Receiving Currency
# money was converted mid-transfer
# This is a classic laundering technique

df['is_currency_mismatch'] = (
    df['Payment Currency'] != df['Receiving Currency']
).astype(int)

print("Currency mismatch count:")
print(df['is_currency_mismatch'].value_counts())

print("\nMismatch in fraud vs normal:")
print(df.groupby('Is Laundering')['is_currency_mismatch'].mean().round(4))

Currency mismatch count:
is_currency_mismatch
0    5006170
1      72166
Name: count, dtype: int64

Mismatch in fraud vs normal:
Is Laundering
0    0.0142
1    0.0000
Name: is_currency_mismatch, dtype: float64


## Check amount values for wierd values

In [25]:
# Check for negative amounts or zero amounts
# These should not exist in normal banking

print("Amount Paid — basic stats:")
print(df['Amount Paid'].describe())

print(f"\nNegative amounts: {(df['Amount Paid'] < 0).sum()}")
print(f"Zero amounts: {(df['Amount Paid'] == 0).sum()}")

Amount Paid — basic stats:
count    5.078336e+06
mean     4.509281e+06
std      8.697736e+08
min      1.000000e-06
25%      1.844800e+02
50%      1.414570e+03
75%      1.229815e+04
max      1.046302e+12
Name: Amount Paid, dtype: float64

Negative amounts: 0
Zero amounts: 0


## Label encoding sender and receiver account

In [29]:
# ============================================
# Label Encoding for Sender & Receiver Accounts
# ============================================

import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Create Label Encoder
le_account = LabelEncoder()

# Combine all accounts from both columns
all_accounts = pd.concat([
    df['Sender Account'],
    df['Receiver Account']
], ignore_index=True)

# Learn all unique account IDs
le_account.fit(all_accounts)

# Encode Sender Account
df['Sender_Account_Encoded'] = le_account.transform(
    df['Sender Account']
)

# Encode Receiver Account
df['Receiver_Account_Encoded'] = le_account.transform(
    df['Receiver Account']
)

# Display sample results
print("Sample Account Encoding:")
print(df[['Sender Account',
          'Sender_Account_Encoded',
          'Receiver Account',
          'Receiver_Account_Encoded']].head())

# Display dataset information
print("\nTotal unique accounts:", len(le_account.classes_))

# Verify data types
print("\nEncoded Column Types:")
print(df[['Sender_Account_Encoded',
          'Receiver_Account_Encoded']].dtypes)

Sample Account Encoding:
  Sender Account  Sender_Account_Encoded Receiver Account  \
0      8000EBD30                     942        8000EBD30   
1      8000F4580                     991        8000F5340   
2      8000F4670                     992        8000F4670   
3      8000F5030                     998        8000F5030   
4      8000F5200                    1000        8000F5200   

   Receiver_Account_Encoded  
0                       942  
1                      1002  
2                       992  
3                       998  
4                      1000  

Total unique accounts: 515080

Encoded Column Types:
Sender_Account_Encoded      int64
Receiver_Account_Encoded    int64
dtype: object


## Flagging outlier in our AML system ⭐⭐⭐⭐⭐

In [ ]:
Two types of outliers in AML
Normal data cleaning  →  outlier = error  →  REMOVE it
AML detection         →  outlier = fraud signal →  FLAG it
Never remove outliers from your AML dataset. A transaction 100x the normal amount is exactly what you want your model to learn from.

### First understand the amount distribution

In [30]:
import matplotlib.pyplot as plt
import numpy as np

# First look at Amount Paid statistics
print("Amount Paid — full statistics:")
print(df['Amount Paid'].describe())

print(f"\nMean   : {df['Amount Paid'].mean():,.2f}")
print(f"Median : {df['Amount Paid'].median():,.2f}")
print(f"Std Dev: {df['Amount Paid'].std():,.2f}")
print(f"Max    : {df['Amount Paid'].max():,.2f}")
print(f"Min    : {df['Amount Paid'].min():,.2f}")

# Compare fraud vs normal amounts
print("\nAverage Amount Paid:")
print(df.groupby('Is Laundering')['Amount Paid'].describe())

Amount Paid — full statistics:
count    5.078336e+06
mean     4.509281e+06
std      8.697736e+08
min      1.000000e-06
25%      1.844800e+02
50%      1.414570e+03
75%      1.229815e+04
max      1.046302e+12
Name: Amount Paid, dtype: float64

Mean   : 4,509,281.36
Median : 1,414.57
Std Dev: 869,773,601.62
Max    : 1,046,302,363,293.48
Min    : 0.00

Average Amount Paid:
                   count          mean           std       min      25%  \
Is Laundering                                                             
0              5073159.0  4.477008e+06  8.688471e+08  0.000001   184.16   
1                 5177.0  3.613531e+07  1.527919e+09  0.003227  2634.97   

                   50%        75%           max  
Is Laundering                                    
0              1410.99  12279.355  1.046302e+12  
1              8667.21  18832.270  8.485314e+10  


### Method-1:IQR outlier detection

In [32]:
# IQR = Interquartile Range
# This is the most standard method for detecting outliers
# 
# How it works:
# Q1 = value below which 25% of data falls
# Q3 = value below which 75% of data falls  
# IQR = Q3 - Q1  (the middle 50% range)
#
# Lower fence = Q1 - 1.5 x IQR  (below this = outlier)
# Upper fence = Q3 + 1.5 x IQR  (above this = outlier)

Q1 = df['Amount Paid'].quantile(0.25)
Q3 = df['Amount Paid'].quantile(0.75)
IQR = Q3 - Q1

lower_fence = Q1 - (1.5 * IQR)
upper_fence = Q3 + (1.5 * IQR)

print(f"Q1 (25th percentile) : {Q1:,.2f}")
print(f"Q3 (75th percentile) : {Q3:,.2f}")
print(f"IQR                  : {IQR:,.2f}")
print(f"Lower fence          : {lower_fence:,.2f}")
print(f"Upper fence          : {upper_fence:,.2f}")

# Flag transactions outside these fences
df['is_amount_outlier_iqr'] = (
    (df['Amount Paid'] < lower_fence) |
    (df['Amount Paid'] > upper_fence)
).astype(int)

print(f"\nOutliers detected    : {df['is_amount_outlier_iqr'].sum():,}")
print(f"Percentage of data   : {df['is_amount_outlier_iqr'].mean()*100:.2f}%")

# How many of these outliers are actually fraud?
print("\nOutlier vs Fraud overlap:")
print(df.groupby(['is_amount_outlier_iqr','Is Laundering']).size())

Q1 (25th percentile) : 184.48
Q3 (75th percentile) : 12,298.15
IQR                  : 12,113.67
Lower fence          : -17,986.03
Upper fence          : 30,468.65

Outliers detected    : 881,712
Percentage of data   : 17.36%

Outlier vs Fraud overlap:
is_amount_outlier_iqr  Is Laundering
0                      0                4192457
                       1                   4167
1                      0                 880702
                       1                   1010
dtype: int64


### Method-2:Z-score outlier detection

In [33]:
# Z-Score tells you how many standard deviations
# a value is away from the mean
#
# Z = (value - mean) / std_deviation
#
# Z > 3  = very unusual (3 standard deviations above mean)
# Z < -3 = very unusual (3 standard deviations below mean)
#
# Example:
# Amount = 500000, Mean = 5000, Std = 8000
# Z = (500000 - 5000) / 8000 = 55.6  ← extremely suspicious!

from scipy import stats

df['amount_zscore'] = np.abs(
    stats.zscore(df['Amount Paid'])
)

# Flag anything with Z-score above 3
df['is_amount_outlier_zscore'] = (
    df['amount_zscore'] > 3
).astype(int)

print("Z-Score outliers:")
print(f"Total flagged : {df['is_amount_outlier_zscore'].sum():,}")
print(f"Percentage    : {df['is_amount_outlier_zscore'].mean()*100:.2f}%")

# Show the most extreme outliers
print("\nTop 5 most extreme transactions by Z-Score:")
top_outliers = df.nlargest(5, 'amount_zscore')[
    ['Amount Paid', 'amount_zscore', 
     'Payment Format', 'Is Laundering']
]
print(top_outliers)

Z-Score outliers:
Total flagged : 985
Percentage    : 0.02%

Top 5 most extreme transactions by Z-Score:
          Amount Paid  amount_zscore Payment Format  Is Laundering
3130043  1.046302e+12    1202.954372            ACH              0
3437493  9.659333e+11    1110.552090            ACH              0
1711894  6.260355e+11     719.763226         Cheque              0
4466474  6.260355e+11     719.763226         Cheque              0
1773261  5.593344e+11     643.075292         Cheque              0


### Method-3:Amount Ratio per sender amount

In [35]:
# This is the most powerful outlier method for AML
# because it compares each transaction to THAT SPECIFIC
# account's own history — not the whole dataset average
#
# A ₹5,00,000 transaction from a business = normal
# A ₹5,00,000 transaction from a student account = very suspicious
#
# amount_ratio = this transaction / account's median transaction

# Calculate each sender account's median transaction amount
account_median = df.groupby('Sender Account')['Amount Paid'].transform('median')
account_std    = df.groupby('Sender Account')['Amount Paid'].transform('std').fillna(1)

# How many times larger is this transaction vs account normal?
df['amount_vs_account_median'] = df['Amount Paid'] / (account_median + 1)

# Flag if transaction is 5x or more than account's own median
df['is_account_level_outlier'] = (
    df['amount_vs_account_median'] > 5
).astype(int)

print("Account-level outliers (5x above own median):")
print(f"Total flagged : {df['is_account_level_outlier'].sum():,}")
print(f"Percentage    : {df['is_account_level_outlier'].mean()*100:.2f}%")

# Most important check — do these overlap with fraud?
print("\nAre account-level outliers actually fraud?")
overlap = df[df['is_account_level_outlier'] == 1]['Is Laundering'].value_counts()
print(overlap)
print(f"\nFraud rate in outliers  : {overlap.get(1,0)/overlap.sum()*100:.2f}%")
print(f"Fraud rate in full data : {df['Is Laundering'].mean()*100:.4f}%")

Account-level outliers (5x above own median):
Total flagged : 770,590
Percentage    : 15.17%

Are account-level outliers actually fraud?
Is Laundering
0    769388
1      1202
Name: count, dtype: int64

Fraud rate in outliers  : 0.16%
Fraud rate in full data : 0.1019%


### Method 4:velocity outlier

In [36]:
# Velocity = how fast is money moving from one account?
# 
# Normal person: 1-2 transactions per day
# Money launderer: 50 transactions in one hour
#
# This catches STRUCTURING — breaking large amounts
# into many small transactions to avoid detection

# Count transactions per sender per hour
df['txn_count_per_sender_hour'] = df.groupby(
    ['Sender Account', 'Hour']
)['Amount Paid'].transform('count')

# Count transactions per sender per day
df['txn_count_per_sender_day'] = df.groupby(
    ['Sender Account', 'Day']
)['Amount Paid'].transform('count')

# Flag high velocity accounts
df['is_high_velocity'] = (
    df['txn_count_per_sender_hour'] > 10   # more than 10 txns per hour
).astype(int)

print("High velocity accounts (10+ txns per hour):")
print(f"Total flagged : {df['is_high_velocity'].sum():,}")

print("\nVelocity outliers vs fraud:")
print(df.groupby('is_high_velocity')['Is Laundering'].mean().round(4))

High velocity accounts (10+ txns per hour):
Total flagged : 732,311

Velocity outliers vs fraud:
is_high_velocity
0    0.001
1    0.001
Name: Is Laundering, dtype: float64


### Combine all outlier flag into one master risk flag

In [37]:
# Create a combined outlier score
# Each flag adds 1 point to the suspicion score
# Maximum score = 4 (flagged by all 4 methods)

df['outlier_score'] = (
    df['is_amount_outlier_iqr']      +   # method 1
    df['is_amount_outlier_zscore']   +   # method 2
    df['is_account_level_outlier']   +   # method 3
    df['is_high_velocity']               # method 4
)

# Create a master flag — suspicious if flagged by ANY method
df['is_outlier'] = (df['outlier_score'] > 0).astype(int)

print("Outlier score distribution:")
print(df['outlier_score'].value_counts().sort_index())

print("\nFraud rate by outlier score:")
fraud_by_score = df.groupby('outlier_score')['Is Laundering'].mean() * 100
for score, rate in fraud_by_score.items():
    bar = '█' * int(rate * 10)
    print(f"  Score {score}: {rate:.3f}% fraud  {bar}")

print(f"\nTotal outlier flagged : {df['is_outlier'].sum():,}")
print(f"Clean transactions   : {(df['is_outlier']==0).sum():,}")

Outlier score distribution:
outlier_score
0    3301810
1    1256373
2     431377
3      88633
4        143
Name: count, dtype: int64

Fraud rate by outlier score:
  Score 0: 0.091% fraud  
  Score 1: 0.118% fraud  █
  Score 2: 0.137% fraud  █
  Score 3: 0.091% fraud  
  Score 4: 0.000% fraud  

Total outlier flagged : 1,776,526
Clean transactions   : 3,301,810


### Add outlier columns to final cleaned dataset

In [38]:
# Update columns_to_keep to include all outlier flags

columns_to_keep = [
    # Time features
    'Hour', 'Day', 'DayOfWeek', 'IsWeekend',

    # Bank features
    'Sender Bank ID', 'Receiver Bank ID', 'is_cross_bank',

    # Account features
    'Sender_Account_Encoded', 'Receiver_Account_Encoded',
    'is_self_transfer',

    # Amount features
    'Amount Paid', 'Amount Received',
    'is_currency_mismatch',
    'amount_vs_account_median',      # how large vs own history

    # Outlier flags ← NEW
    'is_amount_outlier_iqr',         # IQR method
    'is_amount_outlier_zscore',      # Z-score method
    'is_account_level_outlier',      # account history method
    'is_high_velocity',              # velocity method
    'outlier_score',                 # combined score 0-4
    'txn_count_per_sender_hour',     # raw velocity count
    'txn_count_per_sender_day',      # daily count

    # Payment features
    'fmt_ACH', 'fmt_Bitcoin', 'fmt_Cash',
    'fmt_Cheque', 'fmt_Reinvestment', 'fmt_Wire',
    'Payment_Currency_Encoded',
    'Receiving_Currency_Encoded',

    # Target — never drop
    'Is Laundering'
]

df_clean = df[columns_to_keep].copy()

print("Final cleaned dataset shape:", df_clean.shape)
print(f"\nTotal columns: {len(df_clean.columns)}")
print(f"Fraud rows   : {df_clean['Is Laundering'].sum():,}")
print(f"Normal rows  : {(df_clean['Is Laundering']==0).sum():,}")

# Save it
df_clean.to_csv('cleaned_transactions.csv', index=False)
print("\nSaved as cleaned_transactions.csv")

Final cleaned dataset shape: (5078336, 30)

Total columns: 30
Fraud rows   : 5,177
Normal rows  : 5,073,159

Saved as cleaned_transactions.csv


## Build final cleaned dataset

In [39]:
# Get all OHE format columns dynamically
fmt_cols = [c for c in df.columns if c.startswith('fmt_')]

columns_to_keep = (
    # ── Time ─────────────────────────────────────
    ['Hour', 'Day', 'DayOfWeek', 'IsWeekend'] +

    # ── Bank ─────────────────────────────────────
    ['Sender Bank ID', 'Receiver Bank ID',
     'is_cross_bank'] +

    # ── Account ───────────────────────────────────
    ['Sender_Account_Encoded', 'Receiver_Account_Encoded',
     'is_self_transfer'] +

    # ── Amount ────────────────────────────────────
    ['Amount Paid', 'Amount Received',
     'amount_vs_account_median',
     'is_currency_mismatch'] +

    # ── Outlier flags ─────────────────────────────
    ['is_amount_outlier_iqr',
     'is_amount_outlier_zscore',
     'is_account_level_outlier',
     'is_high_velocity',
     'outlier_score',
     'is_outlier',
     'txn_count_per_sender_hour',
     'txn_count_per_sender_day'] +

    # ── Payment format OHE ────────────────────────
    fmt_cols +

    # ── Currency ──────────────────────────────────
    ['Payment_Currency_Encoded',
     'Receiving_Currency_Encoded'] +

    # ── Target ────────────────────────────────────
    ['Is Laundering']
)

df_clean = df[columns_to_keep].copy()

print("═" * 40)
print("FINAL CLEANED DATASET")
print("═" * 40)
print(f"Shape          : {df_clean.shape}")
print(f"Total columns  : {len(df_clean.columns)}")
print(f"Fraud rows     : {df_clean['Is Laundering'].sum():,}")
print(f"Normal rows    : {(df_clean['Is Laundering']==0).sum():,}")
print(f"Missing values : {df_clean.isnull().sum().sum()}")
print("\nAll columns:")
for i, col in enumerate(df_clean.columns, 1):
    print(f"  {i:2}. {col}")

════════════════════════════════════════
FINAL CLEANED DATASET
════════════════════════════════════════
Shape          : (5078336, 32)
Total columns  : 32
Fraud rows     : 5,177
Normal rows    : 5,073,159
Missing values : 0

All columns:
   1. Hour
   2. Day
   3. DayOfWeek
   4. IsWeekend
   5. Sender Bank ID
   6. Receiver Bank ID
   7. is_cross_bank
   8. Sender_Account_Encoded
   9. Receiver_Account_Encoded
  10. is_self_transfer
  11. Amount Paid
  12. Amount Received
  13. amount_vs_account_median
  14. is_currency_mismatch
  15. is_amount_outlier_iqr
  16. is_amount_outlier_zscore
  17. is_account_level_outlier
  18. is_high_velocity
  19. outlier_score
  20. is_outlier
  21. txn_count_per_sender_hour
  22. txn_count_per_sender_day
  23. fmt_ACH
  24. fmt_Bitcoin
  25. fmt_Cash
  26. fmt_Cheque
  27. fmt_Credit Card
  28. fmt_Reinvestment
  29. fmt_Wire
  30. Payment_Currency_Encoded
  31. Receiving_Currency_Encoded
  32. Is Laundering


## Save the final file

In [45]:
df_clean.to_csv(r'C:\Users\tjain\Downloads\HI-Small_Trans.csv', index=False)

print("cleaned_transactions.csv saved!")
print(f"Shape  : {df_clean.shape}")
print(f"Size   : {df_clean.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print("\nNotebook 2 complete!")

cleaned_transactions.csv saved!
Shape  : (5078336, 32)
Size   : 983.1 MB

Notebook 2 complete!
